In [1]:
from peft import PeftModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Configuration ---
base_model_id = "Qwen/Qwen2-0.5B-Instruct"
adapter_path = "./qwen2_npc_adapter_finetuned" # Path where your adapter was saved
# --- --- --- --- ---

# Load the base model tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the base model using float16 instead of bfloat16
print("Loading base model with Float16...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    # torch_dtype=torch.bfloat16, # <--- CHANGE THIS
    torch_dtype=torch.float16,   # <--- TO THIS
    device_map="auto",
    trust_remote_code=True
)
print("Base model loaded.")

# Load the LoRA adapter
print("Loading adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)
print("Adapter loaded.")

# Optional: Merge adapter for faster inference
print("Merging adapter...")
try:
    model = model.merge_and_unload()
    print("Adapter merged.")
except Exception as e:
    print(f"Could not merge and unload adapter (might require more memory): {e}")
    print("Proceeding with unmerged adapter.")

model.eval() # Set model to evaluation mode
print("Fine-tuned model ready for inference.")

# --- Example Inference ---
# (Your inference code follows as before)
# ...

Loading base model with Float16...
Base model loaded.
Loading adapter...
Adapter loaded.
Merging adapter...
Adapter merged.
Fine-tuned model ready for inference.


In [3]:

# --- Try another example ---
npc_personality = "kind"
history = "player punched npc before"
player_holding = "sword"
proximity = 'very close'
instruction = f"NPC Personality: {npc_personality}. History: {history}. Player is holding: {player_holding}. Proximity: {proximity}. Generate NPC response."
instruction = "write a story about secularism"
messages = [
    {"role": "system", "content": "You are an NPC dialogue generation model. Generate a natural, emotionally aligned response based on the provided context."},
    {"role": "user", "content": instruction}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=50, temperature=0.7, top_p=0.9, do_sample=True,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id
    )
response_ids = outputs[0][inputs.input_ids.shape[1]:]
response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
print(f"\nInput Prompt:\n{prompt}")
print(f"\nGenerated Response:\n{response_text}")


Input Prompt:
<|im_start|>system
You are an NPC dialogue generation model. Generate a natural, emotionally aligned response based on the provided context.<|im_end|>
<|im_start|>user
write a story about secularism<|im_end|>
<|im_start|>assistant


Generated Response:
*innards of player's mouth creaks at joke*
